# TGN Ablation Study — 3W Dataset

**Research question:** Does GRU memory and time-projection embedding improve anomaly detection on temporal sensor graphs?

## Three variants (Rossi et al. 2020, Table 2)

| Variant | Memory | Embedding | Role |
|---|---|---|---|
| TGN-no-mem | — | MLP(edge) | Ablation baseline: pure edge features |
| TGN-id | GRU | Identity | Memory, no temporal projection |
| TGN-time (JODIE) | GRU | (1+Δt·w)⊙s | Memory + staleness signal |

## Graph topology: per-instance bipartite
```
well_inst_001 → P-PDG   edge_feat = [sensor_norm, delta_t]
well_inst_001 → P-TPT
             → ...T-PDG
well_inst_002 → P-PDG   (independent GRU memory from inst_001)
```
Each sensor reading is an edge. The GRU memory of each well instance accumulates
only its own history, capturing the temporal evolution of that specific well.

## Split: temporal within-instance (Rossi et al. 2020, §5.1)
- **Train**: first 70 % of each instance's timesteps (chronological order)
- **Test**: last 30 % of each instance's timesteps
- Every instance appears in **both** sets → GRU memory is warmed up before evaluation
- No temporal leakage: test data is always after training data

## 1. Setup

In [1]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import classification_report, roc_auc_score, f1_score
from sklearn.preprocessing import StandardScaler
import sys
sys.path.insert(0, '../../src')

torch.manual_seed(42)
np.random.seed(42)

PROCESSED_DIR = Path('/home/obiaggi/3w_processed')
MODELS_DIR    = Path('/home/obiaggi/TKG_Thesis/models')
MODELS_DIR.mkdir(exist_ok=True, parents=True)

device_str = 'GPU' if torch.cuda.is_available() else 'CPU'
print(f'PyTorch : {torch.__version__}')
print(f'Device  : {device_str}')

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


PyTorch : 2.9.1+cu128
Device  : GPU


## 2. Load 3W Data

In [2]:
SENSOR_COLS = ['P-PDG', 'P-TPT', 'T-TPT', 'P-MON-CKP', 'T-JUS-CKP', 'P-JUS-CKGL', 'QGL', 'T-PDG']
ROWS_PER_CLASS = 10000

CLASS_NAMES = {
    0: 'Normal',          1: 'BSW increase',    2: 'DHSV closure',
    3: 'Severe slugging', 4: 'Flow instability', 5: 'Productivity loss',
    6: 'Quick restriction', 7: 'Scaling',        8: 'Hydrate',
    9: 'Other'
}

dfs = []
for type_num in range(10):
    f = PROCESSED_DIR / f'type_{type_num}.parquet'
    if not f.exists():
        print(f'  Missing type_{type_num} — skipping')
        continue
    d = pd.read_parquet(f)
    if 'event_type' in d.columns and 'class' not in d.columns:
        d = d.rename(columns={'event_type': 'class'})
    if 'instance_id' not in d.columns:
        d['instance_id'] = f'WELL-{type_num:02d}'
    else:
        d['instance_id'] = d['instance_id'].astype(str)
    available = [c for c in SENSOR_COLS if c in d.columns]
    d = d.dropna(subset=available, how='all')
    if len(d) > ROWS_PER_CLASS:
        d = d.sample(ROWS_PER_CLASS, random_state=42)
    d['is_anomaly'] = (d['class'].fillna(0) > 0).astype(int)
    d['type_num'] = type_num
    dfs.append(d)
    n_inst   = d['instance_id'].nunique()
    anom_pct = d['is_anomaly'].mean() * 100
    print(f'  type_{type_num} ({CLASS_NAMES[type_num]:<20}): {len(d):>6,} rows | '
          f'{n_inst:>3} instances | anomaly: {anom_pct:.0f}%')

df = pd.concat(dfs, ignore_index=True)
available_sensors = [c for c in SENSOR_COLS if c in df.columns]

total      = len(df)
n_normal   = (df['is_anomaly'] == 0).sum()
n_anomaly  = (df['is_anomaly'] == 1).sum()
print(f'\nTotal rows    : {total:,}')
print(f'Normal        : {n_normal:,} ({n_normal/total*100:.1f}%)')
print(f'Anomaly       : {n_anomaly:,} ({n_anomaly/total*100:.1f}%)')
print(f'Well instances: {df["instance_id"].nunique()} | Sensors: {len(available_sensors)}')

  type_0 (Normal              ): 10,000 rows | 594 instances | anomaly: 0%


  type_1 (BSW increase        ): 10,000 rows | 128 instances | anomaly: 90%


  type_2 (DHSV closure        ): 10,000 rows |  38 instances | anomaly: 70%


  type_3 (Severe slugging     ): 10,000 rows | 106 instances | anomaly: 97%


  type_4 (Flow instability    ): 10,000 rows | 343 instances | anomaly: 67%


  type_5 (Productivity loss   ): 10,000 rows | 450 instances | anomaly: 97%


  type_6 (Quick restriction   ): 10,000 rows | 221 instances | anomaly: 92%


  type_7 (Scaling             ): 10,000 rows |  46 instances | anomaly: 86%


  type_8 (Hydrate             ): 10,000 rows |  95 instances | anomaly: 84%


  type_9 (Other               ): 10,000 rows | 207 instances | anomaly: 65%

Total rows    : 100,000
Normal        : 25,304 (25.3%)
Anomaly       : 74,696 (74.7%)
Well instances: 1568 | Sensors: 8


## 3. Build Bipartite Graph — Temporal Split

**Why temporal split, not random shuffle?**

Random shuffle would create *temporal leakage*: the model could train on timestep t=800
and then be tested on t=100 — it would have already "seen the future."
The chronological split (first 70% → train, last 30% → test) is the standard in
temporal graph learning (Rossi et al. 2020 §5.1; Kumar et al. JODIE; Pareja et al. EvolveGCN)
and reflects the real deployment scenario: train on past history, predict upcoming anomalies.

In [3]:
def prepare_tgn_data(df):
    sensor_cols = [c for c in SENSOR_COLS if c in df.columns]

    # Melt: each (row, sensor) pair becomes one edge
    melted = (
        df[['instance_id', 'is_anomaly'] + sensor_cols]
        .melt(id_vars=['instance_id', 'is_anomaly'],
              value_vars=sensor_cols,
              var_name='sensor_name', value_name='sensor_value')
        .dropna(subset=['sensor_value'])
        .reset_index(drop=True)
    )

    # Per-instance normalized timestep delta_t in [0,1]
    melted['row_rank'] = melted.groupby('instance_id').cumcount()
    inst_len = melted.groupby('instance_id')['row_rank'].transform('max') + 1
    melted['delta_t'] = melted['row_rank'] / inst_len

    # Global sensor normalization
    scaler = StandardScaler()
    melted['sensor_norm'] = scaler.fit_transform(melted[['sensor_value']])

    # Node index mapping: well instances first, then sensor types
    instances = melted['instance_id'].unique().tolist()
    sensors   = sensor_cols
    node2idx  = {n: i for i, n in enumerate(instances + sensors)}
    num_nodes = len(instances) + len(sensors)

    # --- Temporal within-instance split (Rossi et al. 2020, §5.1) ---
    # Sort by instance and chronological order, then cut at 70th percentile
    melted_s = melted.sort_values(['instance_id', 'row_rank']).reset_index(drop=True)
    at_70 = (
        melted_s.groupby('instance_id')['row_rank']
        .transform(lambda rk: rk.quantile(0.7))
    )
    train_mask = melted_s['row_rank'] <= at_70
    train_df   = melted_s[train_mask]
    test_df    = melted_s[~train_mask]

    def to_tensors(d):
        src = torch.tensor([node2idx[w] for w in d['instance_id']], dtype=torch.long)
        dst = torch.tensor([node2idx[s] for s in d['sensor_name']], dtype=torch.long)
        ef  = torch.tensor(d[['sensor_norm', 'delta_t']].values, dtype=torch.float32)
        dt  = torch.tensor(d['delta_t'].values,               dtype=torch.float32).unsqueeze(1)
        y   = torch.tensor(d['is_anomaly'].values.astype(float), dtype=torch.float32)
        return src, dst, ef, dt, y

    tr_anom = train_df['is_anomaly'].mean() * 100
    te_anom = test_df['is_anomaly'].mean()  * 100
    print(f'Nodes  : {num_nodes} ({len(instances)} well instances + {len(sensors)} sensor types)')
    print(f'Train  : {len(train_df):,} edges  | anomaly rate: {tr_anom:.1f}%')
    print(f'Test   : {len(test_df):,} edges   | anomaly rate: {te_anom:.1f}%')
    print(f'Split  : temporal within-instance (first 70 % train / last 30 % test)')

    return to_tensors(train_df), to_tensors(test_df), num_nodes, instances


train_data, test_data, num_nodes, instances = prepare_tgn_data(df)

Nodes  : 1576 (1568 well instances + 8 sensor types)
Train  : 368,621 edges  | anomaly rate: 74.8%
Test   : 158,443 edges   | anomaly rate: 73.9%
Split  : temporal within-instance (first 70 % train / last 30 % test)


## 4. Train and Evaluate All Variants

Three variants trained in sequence, same data, same hyperparameters.
Results collected in `results` dict for comparison.

In [4]:
from models.tgn import TGN, TGNTime, TGNNoMem

EPOCHS     = 5
BATCH_SIZE = 512
LR         = 1e-3
MEM_DIM    = 64
MSG_DIM    = 64
EMB_DIM    = 64
EDGE_DIM   = 2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training device: {device}')

# Move all data tensors to device once (avoid per-batch .to() overhead)
src_tr_d, dst_tr_d, ef_tr_d, dt_tr_d, y_tr_d = (t.to(device) for t in train_data)
src_te_d, dst_te_d, ef_te_d, dt_te_d, y_te_d = (t.to(device) for t in test_data)

# Class-imbalance weight: upweight minority (Normal)
n_normal_tr  = float((y_tr_d == 0).sum().item())
n_anomaly_tr = float((y_tr_d == 1).sum().item())
normal_weight = n_anomaly_tr / max(n_normal_tr, 1.0)
print(f'Train  — Normal: {int(n_normal_tr):,}  Anomaly: {int(n_anomaly_tr):,}')
print(f'Test   — Normal: {int((y_te_d==0).sum().item()):,}  Anomaly: {int((y_te_d==1).sum().item()):,}')
print(f'Class weight — Normal: {normal_weight:.2f}x  Anomaly: 1.00x\n')

VARIANTS = [
    ('TGN-no-mem', TGNNoMem, False),
    ('TGN-id',     TGN,      True),
    ('TGN-time',   TGNTime,  True),
]

results = {}

for variant_name, ModelCls, embed_src in VARIANTS:
    sep = '=' * 60
    print(f'\n{sep}')
    print(f'  Variant: {variant_name}')
    print(sep)

    model = ModelCls(num_nodes=num_nodes, memory_dim=MEM_DIM,
                     message_dim=MSG_DIM, embed_dim=EMB_DIM, edge_dim=EDGE_DIM).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f'  Parameters: {n_params:,}')

    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    criterion = nn.BCELoss(reduction='none')

    for epoch in range(EPOCHS):
        model.train()
        if hasattr(model, 'memory'):
            model.memory.reset()
        total_loss, n_batches = 0.0, 0
        for i in range(0, len(src_tr_d), BATCH_SIZE):
            s  = src_tr_d[i:i+BATCH_SIZE];  d  = dst_tr_d[i:i+BATCH_SIZE]
            e  = ef_tr_d[i:i+BATCH_SIZE];   t_ = dt_tr_d[i:i+BATCH_SIZE]
            yb = y_tr_d[i:i+BATCH_SIZE]
            optimizer.zero_grad()
            pred = model(s, d, e, t_, update_memory=True, embed_src=embed_src)
            w = torch.where(yb == 0,
                            torch.full_like(yb, normal_weight),
                            torch.ones_like(yb))
            loss = (criterion(pred, yb) * w).mean()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
            n_batches  += 1
        avg_loss = total_loss / n_batches
        print(f'  Epoch {epoch+1}/{EPOCHS} — loss: {avg_loss:.4f}')

    # --- Evaluation (memory from last training epoch is preserved) ---
    model.eval()
    all_scores = []
    with torch.no_grad():
        for i in range(0, len(src_te_d), BATCH_SIZE):
            s = model(src_te_d[i:i+BATCH_SIZE], dst_te_d[i:i+BATCH_SIZE],
                      ef_te_d[i:i+BATCH_SIZE],  dt_te_d[i:i+BATCH_SIZE],
                      update_memory=False, embed_src=embed_src)
            all_scores.extend(s.cpu().numpy())

    scores = np.array(all_scores)
    y_np   = y_te_d.cpu().numpy()

    # Threshold sweep — maximise macro F1
    best_t, best_f1 = 0.5, 0.0
    for t in np.arange(0.05, 0.95, 0.01):
        mf1 = f1_score(y_np, (scores > t).astype(int), average='macro', zero_division=0)
        if mf1 > best_f1:
            best_f1, best_t = mf1, t

    y_pred = (scores > best_t).astype(int)
    try:
        auc = roc_auc_score(y_np, scores)
    except Exception:
        auc = float('nan')

    rep = classification_report(y_np, y_pred,
                                target_names=['Normal', 'Anomaly'],
                                output_dict=True, zero_division=0)

    results[variant_name] = {
        'auc':        auc,
        'macro_f1':   best_f1,
        'threshold':  best_t,
        'normal_f1':  rep['Normal']['f1-score'],
        'normal_rec': rep['Normal']['recall'],
        'anomaly_f1': rep['Anomaly']['f1-score'],
        'n_params':   n_params,
    }

    print(f'\n  Best threshold : {best_t:.2f}  |  macro F1 : {best_f1:.3f}  |  AUC-ROC : {auc:.4f}')
    print(classification_report(y_np, y_pred,
                                target_names=['Normal', 'Anomaly'],
                                digits=3, zero_division=0))

    safe_name = variant_name.replace(' ', '_').replace('(', '').replace(')', '').lower()
    torch.save({'model_state': model.state_dict(), 'num_nodes': num_nodes,
                'variant': variant_name, 'dataset': '3W',
                'split': 'temporal-within-instance'},
               MODELS_DIR / f'tgn_3w_{safe_name}.pt')
    print(f'  Checkpoint saved: tgn_3w_{safe_name}.pt')

print('\nAll variants trained.')

Training device: cuda


Train  — Normal: 92,961  Anomaly: 275,660
Test   — Normal: 41,276  Anomaly: 117,167
Class weight — Normal: 2.97x  Anomaly: 1.00x


  Variant: TGN-no-mem
  Parameters: 6,529


  Epoch 1/5 — loss: 0.9432


  Epoch 2/5 — loss: 1.0634


  Epoch 3/5 — loss: 1.0532


...

  Epoch 2/5 — loss: 1.0124


  Epoch 3/5 — loss: 0.9616


  Epoch 4/5 — loss: 0.9915


  Epoch 5/5 — loss: 0.9671



  Best threshold : 0.50  |  macro F1 : 0.654  |  AUC-ROC : 0.7116
              precision    recall  f1-score   support

      Normal      0.493     0.477     0.485     41276
     Anomaly      0.818     0.827     0.822    117167

    accuracy                          0.736    158443
   macro avg      0.655     0.652     0.654    158443
weighted avg      0.733     0.736     0.735    158443

  Checkpoint saved: tgn_3w_tgn-time.pt

All variants trained.


## 5. Comparison Table

In [5]:
rows = []
for name, r in results.items():
    has_mem = 'No'  if name == 'TGN-no-mem' else 'GRU'
    emb     = 'None'     if name == 'TGN-no-mem' else \
              'Time-proj' if name == 'TGN-time'   else 'Identity'
    rows.append({
        'Variant':    name,
        'Memory':     has_mem,
        'Embedding':  emb,
        'AUC-ROC':    round(r['auc'],       4),
        'Macro F1':   round(r['macro_f1'],  3),
        'Normal F1':  round(r['normal_f1'], 3),
        'Normal Rec': round(r['normal_rec'],3),
        'Anomaly F1': round(r['anomaly_f1'],3),
        'Threshold':  round(r['threshold'], 2),
        'Params':     r['n_params'],
    })

cmp = pd.DataFrame(rows)
cmp = cmp.sort_values('AUC-ROC', ascending=False)

print('TGN Ablation — 3W Dataset (per-instance bipartite, temporal split)')
print('=' * 90)
print(cmp.to_string(index=False))
print()

# Delta analysis
if 'TGN-no-mem' in results and 'TGN-id' in results:
    mem_gain = results['TGN-id']['auc'] - results['TGN-no-mem']['auc']
    print(f'GRU memory contribution  (TGN-id   vs TGN-no-mem): ΔAUC = {mem_gain:+.4f}')

if 'TGN-id' in results and 'TGN-time' in results:
    time_gain = results['TGN-time']['auc'] - results['TGN-id']['auc']
    print(f'Time-projection gain     (TGN-time vs TGN-id):      ΔAUC = {time_gain:+.4f}')

best = max(results, key=lambda k: results[k]['auc'])
print(f'\nBest variant: {best}  (AUC = {results[best]["auc"]:.4f}, macro F1 = {results[best]["macro_f1"]:.3f})')

TGN Ablation — 3W Dataset (per-instance bipartite, temporal split)
   Variant Memory Embedding  AUC-ROC  Macro F1  Normal F1  Normal Rec  Anomaly F1  Threshold  Params
  TGN-time    GRU Time-proj   0.7116     0.654      0.485       0.477       0.822       0.50  149057
    TGN-id    GRU  Identity   0.7110     0.654      0.485       0.477       0.822       0.49  148993
TGN-no-mem     No      None   0.5031     0.425      0.000       0.000       0.850       0.05    6529

GRU memory contribution  (TGN-id   vs TGN-no-mem): ΔAUC = +0.2079
Time-projection gain     (TGN-time vs TGN-id):      ΔAUC = +0.0006

Best variant: TGN-time  (AUC = 0.7116, macro F1 = 0.654)


## 6. Insights for Thesis

### What to read from the results

**If ΔAUC(TGN-id vs TGN-no-mem) > 0.02:**
The GRU memory meaningfully contributes: the model is learning the temporal evolution
of each well's sensor state, not just reacting to instantaneous edge features.
This justifies the TGN architecture over a simple MLP classifier.

**If ΔAUC(TGN-time vs TGN-id) > 0.01:**
The time-projection `(1+Δt·w)⊙s` adds value: the model benefits from knowing how
*stale* the node state is (i.e., how much time has passed since the last interaction).
This is consistent with JODIE (Kumar et al. 2019) findings on user-item interaction data.

**If both deltas ≈ 0:**
The anomaly signal on 3W is captured almost entirely by the instantaneous sensor
readings (edge features), and memory adds little. This is a valid and publishable finding:
it suggests the 3W anomaly classes are *stationary* (each sensor reading is discriminative
by itself) rather than *evolving* (requiring historical context).

### Comparison with degenerate graph (04a_tgn_degenerate.ipynb)
The degenerate graph (all wells → single SENSOR node) produces AUC ≈ 0.50 regardless
of the memory variant. Combined with the per-instance results here, this confirms:
- **Graph topology matters** more than the model variant
- Preserving per-instance identity is a prerequisite for any meaningful temporal learning

### Limitation
With ROWS_PER_CLASS=2000 (capped for compute), each well type has at most one instance
(WELL-00…WELL-09) unless the parquet files contain multiple real well IDs.
If `instance_id` exists in the parquet files, the model benefits from multiple
independent trajectories per anomaly class — a richer training signal.